# Lab 7: QEC Circuit Simulation in stim

**[EQE5006] Quantum Error Correction, Spring 2026**

In this lab, we learn how to implement and simulate QEC circuits with Stim.

## Table of contents

0. [Recap](#sec-0)
1. [QEC simulation of the 3-qubit bit-flip code](#sec-1)
2. [The $[[5,1,2]]$ surface code in stim](#sec-2)
3. [Scaling to larger code distances $d \geq 3$](#sec-3)

In [ ]:
# Install stim and PyMatching if not already available
!pip install stim pymatching -q

In [ ]:
import stim
import numpy as np
import matplotlib.pyplot as plt

<a id="sec-0"></a>

---

## Section 0: Recap

In [ ]:
# Reminder 1: bit-flip noise channel and the sampler.
c = stim.Circuit()
c.append("R", [0])
c.append("X_ERROR", [0], 0.15)
c.append("M", [0])

In [ ]:
c.diagram("timeline-svg")

In [ ]:
samples = c.compile_sampler().sample(shots=200_000)
print(samples.mean())

In [ ]:
# Reminder 2: ancilla parity readout.
c = stim.Circuit()
c.append("H", [0])
c.append("CNOT", [0, 1])
c.append("X_ERROR", [0], 0.1)
c.append("R", [2])
c.append("CNOT", [0, 2])
c.append("CNOT", [1, 2])
c.append("M", [2])

In [ ]:
c.diagram("timeline-svg")

In [ ]:
rate = c.compile_sampler().sample(shots=100_000)[:, 0].mean()
print(rate)

<a id="sec-1"></a>

---

## Section 1: QEC simulation of the 3-qubit bit-flip code

### Memory-$Z$ circuit with a single syndrome extraction round under bit-flip noise

1. Prepare $|0_L\rangle = |000\rangle$ on data qubits
2. Apply bit-flip noise `X_ERROR(p)` to all data qubits
3. Apply a syndrome extraction circuit
4. Measure $Z$ on all data qubits $\longrightarrow$ Obtain the value of $\bar{Z} = Z_0$

In [ ]:
c = stim.Circuit()
# Encode |0_L> = |000>
c.append("R", [0, 1, 2])
# Bit-flip noise on data qubits
c.append("X_ERROR", [0, 1, 2], 0.01)
# Syndrome extraction (perfect ancillas)
c.append("R", [3, 4])
c.append("CNOT", [0, 3])
c.append("CNOT", [1, 3])
c.append("CNOT", [1, 4])
c.append("CNOT", [2, 4])
c.append("M", [3, 4])
# Final data measurement (Z-basis).
c.append("M", [0, 1, 2])

In [ ]:
c.diagram("timeline-svg")

### Detectors and observables

- Check $\longrightarrow$ `DETECTOR`
  * `False` when $+1$, `True` when $-1$
  * If `True`, we say the detector **"fires"** or is **"violated"** (**"detection event"**)
- Logical Pauli measurement $\longrightarrow$ `OBSERVABLE_INCLUDE`
  * In the $Z$-memory circuit, `OBSERVABLE_INCLUDE` includes only $\bar{Z}$
  * `False` when $+1$, `True` when $-1$

In [ ]:
def make_bit_flip_with_detectors(p):
    c = stim.Circuit()
    # Encode |0_L> = |000>
    c.append("R", [0, 1, 2])
    # Bit-flip noise on data qubits
    c.append("X_ERROR", [0, 1, 2], p)
    # Syndrome extraction (perfect ancillas)
    c.append("R", [3, 4])
    c.append("CNOT", [0, 3])
    c.append("CNOT", [1, 3])
    c.append("CNOT", [1, 4])
    c.append("CNOT", [2, 4])
    c.append("M", [3, 4])
    # rec[-2] = Z_0 Z_1 syndrome bit; rec[-1] = Z_1 Z_2 syndrome bit.
    c.append("DETECTOR", [stim.target_rec(-2)])
    c.append("DETECTOR", [stim.target_rec(-1)])
    # Final data measurement (Z-basis).
    c.append("M", [0, 1, 2])
    # Logical Z = Z_0 (rec[-3] is q0).
    c.append("OBSERVABLE_INCLUDE", [stim.target_rec(-3)], 0)
    return c


circuit = make_bit_flip_with_detectors(0.1)
print(circuit)

In [ ]:
circuit.diagram("timeline-svg")

In [ ]:
circuit = make_bit_flip_with_detectors(0)
shots = 100_000
sampler = circuit.compile_detector_sampler()
det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)

### Look-up table decoder

The decoder's job is to predict whether the observable should be flipped, given the syndrome (detector outcomes)

For the bit flip code there are only four syndrome patterns:

| $(m_{01},\, m_{12})$ | $(b_{01},\, b_{12})$ | Diagnosis | Predicted observable ($\bar{Z} = Z_1$) flip |
|:---:|:---:|:---|:---:|
| $(+1, +1)$ | `(False, False)` | No error | `False` |
| $(-1, +1)$ | `(True, False)` | $X$ on $q_0$ | `True` |
| $(-1, -1)$ | `(True, True)` | $X$ on $q_1$ | `False` |
| $(+1, -1)$ | `(False, True)` | $X$ on $q_2$ | `False` |

In [ ]:
def lookup_decode(det_events):
    """Lookup-table decoder for the single-round bit flip code.

    Parameters
    ----------
    det_events : np.ndarray
        Boolean array of shape (shots, 2) with columns [s_01, s_12].

    Returns
    -------
    np.ndarray
        Boolean array of shape (shots, 1) — predicted observable flip per shot.
    """
    b01 = det_events[:, 0]
    b12 = det_events[:, 1]
    return (b01 & ~b12).reshape(-1, 1)


preds = lookup_decode(det_events)
obs_flips_corrected = obs_flips ^ preds

In [ ]:
obs_flips_corrected

In [ ]:
print(obs_flips_corrected.mean())

### Simulation with varying $p$

In [ ]:
plist = np.logspace(-2, 0, 11)
p_logical_values = []
p_logical_no_qec_values = []

for p in plist:
    circuit = make_bit_flip_with_detectors(p)
    shots = 100000
    sampler = circuit.compile_detector_sampler()
    det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)
    obs_flips_corrected = obs_flips ^ lookup_decode(det_events)

    p_logical_no_qec = obs_flips.mean()
    p_logical = obs_flips_corrected.mean()
    p_logical_no_qec_values.append(p_logical_no_qec)
    p_logical_values.append(p_logical)

In [ ]:
plt.plot(plist, p_logical_values, "o-", label="QEC")
plt.plot(plist, p_logical_no_qec_values, "^-", label="no QEC")
plt.plot(plist, plist, "k--", alpha=0.5, label="break-even ($p_L = p$)")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate $p_L$")
plt.title("Logical error rate vs physical error rate for the bit flip code")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.show()

<a id="sec-2"></a>

---

## Section 2: The $[[5,1,2]]$ Surface Code in stim

Lecture 09 introduced the smallest surface-code patch:

<center>
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 468 468" width="320" height="320">
<rect width="100%" height="100%" fill="white"/>
<style>text{font-family:Helvetica,Arial,sans-serif;font-size:22px;dominant-baseline:middle;text-anchor:middle}.label{font-size:18px;fill:#111}</style>
<line x1="99" y1="99" x2="369" y2="99" stroke="#c93535" stroke-width="5" stroke-linecap="round"/>
<line x1="99" y1="99" x2="99" y2="369" stroke="#2563eb" stroke-width="5" stroke-linecap="round"/>
<line x1="99" y1="234" x2="369" y2="234" stroke="#2563eb" stroke-width="5" stroke-linecap="round"/>
<line x1="234" y1="99" x2="234" y2="369" stroke="#c93535" stroke-width="5" stroke-linecap="round"/>
<line x1="99" y1="369" x2="369" y2="369" stroke="#c93535" stroke-width="5" stroke-linecap="round"/>
<line x1="369" y1="99" x2="369" y2="369" stroke="#2563eb" stroke-width="5" stroke-linecap="round"/>
<circle cx="99" cy="99" r="27" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="129" y="71">D1</text>
<circle cx="369" cy="99" r="27" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="399" y="71">D2</text>
<circle cx="234" cy="234" r="27" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="264" y="206">D3</text>
<circle cx="99" cy="369" r="27" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="129" y="341">D4</text>
<circle cx="369" cy="369" r="27" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="399" y="341">D5</text>
<rect x="216.5" y="81.5" width="35.1" height="35.1" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="269" y="75">A1</text>
<rect x="81.5" y="216.5" width="35.1" height="35.1" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="134" y="210">A2</text>
<rect x="351.5" y="216.5" width="35.1" height="35.1" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="404" y="210">A3</text>
<rect x="216.5" y="351.5" width="35.1" height="35.1" fill="#ffffff" stroke="#111" stroke-width="3"/>
<text class="label" x="269" y="345">A4</text>
</svg>
</center>

Mapping the labels onto stim qubits ($D_i \to q_{i-1}$ for data, ancilla mapping shown in the table):

| Stabilizer | Type | Support | Figure ancilla | stim ancilla |
|:---:|:---:|:---|:---:|:---:|
| $g_1 = X_{D_1} X_{D_2} X_{D_3}$ | $X$ | $q_0, q_1, q_2$ | $A_1$ | $q_5$ |
| $g_2 = X_{D_3} X_{D_4} X_{D_5}$ | $X$ | $q_2, q_3, q_4$ | $A_4$ | $q_6$ |
| $g_3 = Z_{D_2} Z_{D_3} Z_{D_5}$ | $Z$ | $q_1, q_2, q_4$ | $A_3$ | $q_7$ |
| $g_4 = Z_{D_1} Z_{D_3} Z_{D_4}$ | $Z$ | $q_0, q_2, q_3$ | $A_2$ | $q_8$ |

Logical operators: $\bar{X} = X_{D_1} X_{D_4} = X_0 X_3$, $\;\bar{Z} = Z_{D_1} Z_{D_2} = Z_0 Z_1$.

### Memory-$Z$ circuit with a single syndrome extraction round under bit-flip noise

1. Prepare $|0_L\rangle$ on data qubits
2. Apply bit-flip noise `X_ERROR(p)` to all data qubits
3. Apply a syndrome extraction circuit $\longrightarrow \{ m_{A_i} = \pm 1\}$
4. Measure $\bar{Z} = Z_0 Z_1$

### How to prepare $|0_L\rangle$

1. Prepare $|00000\rangle$ on data qubits
2. Apply a syndrome extraction circuit $\longrightarrow \{ m_{A_i}^{(0)} = \pm 1 \}$

**Note:** This process may give $-1$ for X-type checks ($g_1$, $g_2$). To handle them, define detectors as the product of sequential measurement results:

$$ \mathrm{Detector}(A_i) = m_{A_i}^{(0)} m_{A_i} \longleftrightarrow b_{A_i}^{(0)} \oplus b_{A_i} $$

where $\oplus$ stands for XOR.

In [ ]:
def make_512_memory_z(p):
    c = stim.Circuit()

    # ===== Qubit coordinates (for visualization) =====
    c.append("QUBIT_COORDS", [0], [0, 0])  # D_1
    c.append("QUBIT_COORDS", [1], [2, 0])  # D_2
    c.append("QUBIT_COORDS", [2], [1, 1])  # D_3
    c.append("QUBIT_COORDS", [3], [0, 2])  # D_4
    c.append("QUBIT_COORDS", [4], [2, 2])  # D_5
    c.append("QUBIT_COORDS", [5], [1, 0])  # A_1 (X-stab g_1)
    c.append("QUBIT_COORDS", [6], [1, 2])  # A_4 (X-stab g_2)
    c.append("QUBIT_COORDS", [7], [2, 1])  # A_3 (Z-stab g_3)
    c.append("QUBIT_COORDS", [8], [0, 1])  # A_2 (Z-stab g_4)

    # ===== |00000> on data qubits =====
    c.append("R", [0, 1, 2, 3, 4])

    # ===== Syndrome extraction (for |0_L>) =====
    c.append("R", [7, 8])  # Z-type ancillas (g_3, g_4).
    c.append("RX", [5, 6])  # X-type ancillas (g_1, g_2).
    c.append("TICK")

    for q in [0, 1, 2]:
        c.append("CNOT", [5, q])
        c.append("TICK")

    for q in [2, 3, 4]:
        c.append("CNOT", [6, q])
        c.append("TICK")

    for q in [1, 2, 4]:
        c.append("CNOT", [q, 7])
        c.append("TICK")

    for q in [0, 2, 3]:
        c.append("CNOT", [q, 8])
        c.append("TICK")

    c.append("MX", [5, 6])
    c.append("M", [7, 8])

    # ===== Bit-flip noise on data qubits =====
    c.append("X_ERROR", [0, 1, 2, 3, 4], p)
    c.append("TICK")

    # ===== Syndrome extraction (for |0_L>) =====
    c.append("R", [7, 8])  # Z-type ancillas (g_3, g_4).
    c.append("RX", [5, 6])  # X-type ancillas (g_1, g_2).
    c.append("TICK")

    for q in [0, 1, 2]:
        c.append("CNOT", [5, q])
        c.append("TICK")

    for q in [2, 3, 4]:
        c.append("CNOT", [6, q])
        c.append("TICK")

    for q in [1, 2, 4]:
        c.append("CNOT", [q, 7])
        c.append("TICK")

    for q in [0, 2, 3]:
        c.append("CNOT", [q, 8])
        c.append("TICK")

    c.append("MX", [5, 6])
    c.append("M", [7, 8])

    # Detectors
    c.append("DETECTOR", [stim.target_rec(-1), stim.target_rec(-5)])
    c.append("DETECTOR", [stim.target_rec(-2), stim.target_rec(-6)])
    c.append("DETECTOR", [stim.target_rec(-3), stim.target_rec(-7)])
    c.append("DETECTOR", [stim.target_rec(-4), stim.target_rec(-8)])

    # Final Z-basis measurement of the data qubits and the observable Z_0 Z_1
    c.append("M", [0, 1, 2, 3, 4])
    c.append("OBSERVABLE_INCLUDE", [stim.target_rec(-5), stim.target_rec(-4)], 0)
    return c

In [ ]:
# Inspect the structure at p = 0.05.
c = make_512_memory_z(p=0.05)
print(c)

In [ ]:
c.diagram("timeline-svg")

In [ ]:
c.diagram("timeslice-svg")

### Decoding with PyMatching

The hand-written lookup decoder of Section 1 worked because the bit flip code has only four syndrome patterns. 

The $[[5,1,2]]$ patch already has $2^4 = 16$ patterns, and a distance-$d$ surface code patch grows combinatorially. Writing one rule per syndrome is no longer practical.

A standard decoder for the surface code is the **minimum-weight perfect matching (MWPM) decoder**, implemented by **PyMatching**. 

We will cover this in future lectures. For now, let's just use it.

In [ ]:
import pymatching


def decode(det_events):
    dem = circuit.detector_error_model(decompose_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    preds = matcher.decode_batch(det_events)
    return preds


circuit = make_512_memory_z(p=0.01)
shots = 10000
sampler = circuit.compile_detector_sampler()
det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)
preds = decode(det_events)
obs_flips_corrected = obs_flips ^ preds
print(obs_flips_corrected.mean())

### Full QEC Simulation

In [ ]:
plist = np.logspace(-2, -0.5, 11)
p_logical_values = []
p_logical_no_qec_values = []

for p in plist:
    circuit = make_512_memory_z(p)
    shots = 100000
    sampler = circuit.compile_detector_sampler()
    det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)
    obs_flips_corrected = obs_flips ^ decode(det_events)

    p_logical_no_qec = obs_flips.mean()
    p_logical = obs_flips_corrected.mean()
    p_logical_no_qec_values.append(p_logical_no_qec)
    p_logical_values.append(p_logical)

In [ ]:
plt.plot(plist, p_logical_values, "o-", label="QEC")
plt.plot(plist, p_logical_no_qec_values, "^-", label="no QEC")
plt.plot(plist, plist, "k--", alpha=0.5, label="break-even ($p_L = p$)")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate $p_L$")
plt.title("Logical error rate vs physical error rate for the bit flip code")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.show()

<a id="sec-3"></a>

## Section 3: Scaling to larger code distances $d \geq 3$

In [ ]:
c = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    distance=3,
    rounds=1,
    before_round_data_depolarization=0.01,
)

In [ ]:
c.diagram("timeline-svg")

In [ ]:
c.diagram("timeslice-svg")

In [ ]:
plist = np.logspace(-2, -0.5, 11)
dlist = [3, 5, 7, 9]
p_logical_values = np.zeros((len(dlist), len(plist)))

for idx_d, d in enumerate(dlist):
    for idx_p, p in enumerate(plist):
        # Bit-flip error rate of p can be effectively converted to depolarization error rate of (3/2)p
        circuit = stim.Circuit.generated(
            "surface_code:rotated_memory_z",
            distance=d,
            rounds=1,
            before_round_data_depolarization=p * 1.5,
        )
        shots = 100000
        sampler = circuit.compile_detector_sampler()
        det_events, obs_flips = sampler.sample(shots=shots, separate_observables=True)
        obs_flips_corrected = obs_flips ^ decode(det_events)

        p_logical_values[idx_d, idx_p] = obs_flips_corrected.mean()

In [ ]:
for idx_d, d in enumerate(dlist):
    plt.plot(plist, p_logical_values[idx_d], "o-", label=f"d={d}")

plt.plot(plist, plist, "k--", alpha=0.5, label="break-even ($p_L = p$)")
plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate $p_L$")
plt.title("Logical error rate vs physical error rate for the bit flip code")
plt.xscale("log")
plt.yscale("log")
plt.legend()
plt.show()